In [2]:
%matplotlib inline
import re
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

# Параметры — проверь путь к файлу
INPUT_XLSX = Path("Бровки_now.xlsx")
assert INPUT_XLSX.exists(), f"Файл не найден: {INPUT_XLSX}"
OUTPUT_DIR = Path("results_models")
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "profiles").mkdir(exist_ok=True)
(OUTPUT_DIR / "plots").mkdir(exist_ok=True)
(OUTPUT_DIR / "models").mkdir(exist_ok=True)

# Параметры формата (жёсткие под твой единый формат)
HEADER_ROW_IDX = 1      # индексация с 0: вторая строка Excel содержит заголовки
DATA_START_ROW = 2      # третья строка Excel — первые данные
COLS_PER_PROFILE = 5    # число колонок в одном профиле: Дата измерения, ГП, ИГП, РГП-ПН, ИПН
MAX_EMPTY_RUN = 3
print("Input:", INPUT_XLSX)
print("Outputs ->", OUTPUT_DIR.resolve())


Input: Бровки_now.xlsx
Outputs -> C:\Users\user\Desktop\Diplom\po\results_models


In [3]:
def safe_str(x):
    return "" if pd.isna(x) else str(x).strip()

def clean_num_series(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace(r'\s+', '', regex=True)
         .str.replace('*', '', regex=False)
         .str.replace(',', '.', regex=False)
         .str.replace('−', '-', regex=False)
         .str.replace(r'[^0-9\.\-]', '', regex=True),
        errors='coerce'
    )

def make_unique(names):
    cnt = Counter()
    out = []
    for n in names:
        key = "" if n is None else str(n)
        if cnt[key] == 0:
            out.append(key)
        else:
            out.append(f"{key}__{cnt[key]}")
        cnt[key] += 1
    return out


In [4]:
def parse_sheet_fixed(sheet_df,
                      header_row_idx=HEADER_ROW_IDX,
                      data_start_row=DATA_START_ROW,
                      cols_per_profile=COLS_PER_PROFILE,
                      max_empty_run=MAX_EMPTY_RUN):
    """
    Возвращает (profiles_dict, combined_df) для данного листа.
    profiles_dict: {profile_title: df}
    combined_df: объединение всех profile dfs с колонкой 'Профиль'
    """
    ncols = sheet_df.shape[1]
    n_profiles = ncols // cols_per_profile
    profiles = {}
    combined_parts = []
    # строка 0 может содержать заголовки вида "ПРОФИЛЬ №1" — используем если есть
    title_row = sheet_df.iloc[0] if sheet_df.shape[0] > 0 else None

    expected_headers = ['Дата измерения','ГП','ИГП, м','РГП-ПН, м','ИПН, м']

    for i in range(n_profiles):
        start = i*cols_per_profile
        end = min(start + cols_per_profile, ncols)
        block = sheet_df.iloc[:, start:end].copy()

        # получаем заголовки из строки header_row_idx (вторая строка)
        raw_colnames = block.iloc[header_row_idx].astype(str).fillna("").tolist()
        if all(safe_str(x) == "" for x in raw_colnames):
            raw_colnames = expected_headers[:block.shape[1]]
        # нормализуем на короткие имена по содержимому
        norm_names = []
        for j, nm in enumerate(raw_colnames):
            s = safe_str(nm).lower()
            if 'дата' in s:
                norm_names.append('Дата')
            elif 'гп' == s or s.strip() == 'гп':
                norm_names.append('ГП')
            elif 'игп' in s or 'измер' in s:
                norm_names.append('ИГП_м')
            elif 'ргп' in s or 'пн' in s:
                norm_names.append('РГП_ПН_м')
            elif 'ипн' in s or 'бров' in s:
                norm_names.append('ИПН_м')
            else:
                # fallback by position
                mapping = {0:'Дата',1:'ГП',2:'ИГП_м',3:'РГП_ПН_м',4:'ИПН_м'}
                norm_names.append(mapping.get(j, f"col_{j}"))
        norm_names = make_unique(norm_names)
        block.columns = norm_names

        # Берём данные начиная со строки data_start_row (индекс)
        data_rows = []
        empty_run = 0
        for r in range(data_start_row, sheet_df.shape[0]):
            row = block.iloc[r].values.tolist()
            if all(safe_str(x) == "" for x in row):
                empty_run += 1
                if empty_run >= max_empty_run:
                    break
                else:
                    continue
            empty_run = 0
            data_rows.append(row)
        if not data_rows:
            df = pd.DataFrame(columns=block.columns)
        else:
            df = pd.DataFrame(data_rows, columns=block.columns)

        # Приведение типов
        # Дата
        date_col = next((c for c in df.columns if 'дата' in c.lower()), None)
        if date_col is None:
            # попытка: первая колонка, в которой можно распознать даты
            for c in df.columns[:min(3, len(df.columns))]:
                if pd.to_datetime(df[c], errors='coerce', dayfirst=True).notna().sum() > 0:
                    date_col = c; break
        if date_col:
            df['Дата'] = pd.to_datetime(df[date_col], errors='coerce', dayfirst=True)
        else:
            df['Дата'] = pd.NaT

        # Числовые поля
        # ИГП_м
        if 'ИГП_м' in df.columns:
            df['ИГП_м'] = clean_num_series(df['ИГП_м'])
        else:
            for c in df.columns:
                if 'игп' in c.lower() or 'измер' in c.lower():
                    df['ИГП_м'] = clean_num_series(df[c]); break
        # РГП_ПН_м
        if 'РГП_ПН_м' in df.columns:
            df['РГП_ПН_м'] = clean_num_series(df['РГП_ПН_м'])
        else:
            for c in df.columns:
                if 'ргп' in c.lower() or ('пн' in c.lower() and 'ргп' not in c.lower()):
                    df['РГП_ПН_м'] = clean_num_series(df[c]); break
        # ИПН_м
        if 'ИПН_м' in df.columns:
            df['ИПН_м'] = clean_num_series(df['ИПН_м'])
        else:
            for c in df.columns:
                if 'ипн' in c.lower() or 'бров' in c.lower():
                    df['ИПН_м'] = clean_num_series(df[c]); break
        # ГП / Пункт
        if 'ГП' in df.columns:
            df['ГП'] = df['ГП'].astype(str).str.strip().replace("nan","")
        # Дроп строк без даты
        if 'Дата' in df.columns:
            df = df.dropna(subset=['Дата']).reset_index(drop=True)
        else:
            df = df.iloc[0:0]

        # Год
        if 'Дата' in df.columns and not df['Дата'].isna().all():
            df['Год'] = df['Дата'].dt.year
        else:
            df['Год'] = np.nan

        # название профиля: если в title_row есть текст в start col — используем, иначе "Профиль_i"
        profile_title = None
        try:
            if title_row is not None:
                v = safe_str(title_row.iloc[start])
                if v:
                    profile_title = v
        except Exception:
            profile_title = None
        if not profile_title:
            profile_title = f"Профиль_{i+1}"

        df_profile = df[[c for c in ['Дата','Пункт','ГП','ИГП_м','РГП_ПН_м','ИПН_м','Год'] if c in df.columns]].copy()
        profiles[profile_title] = df_profile
        cp = df_profile.copy()
        cp['Профиль'] = profile_title
        combined_parts.append(cp)

    combined = pd.concat(combined_parts, ignore_index=True) if combined_parts else pd.DataFrame()
    return profiles, combined


In [5]:
xls = pd.ExcelFile(INPUT_XLSX)
sites_dict = {}
for sheet in xls.sheet_names:
    raw = pd.read_excel(INPUT_XLSX, sheet_name=sheet, header=None)
    profiles, combined = parse_sheet_fixed(raw)
    if not combined.empty:
        combined['Участок'] = sheet
    sites_dict[sheet] = {'profiles': profiles, 'combined': combined}

# Объединённая таблица всех наблюдений
df_observations = pd.concat([v['combined'] for v in sites_dict.values() if not v['combined'].empty], ignore_index=True)
df_observations.to_excel(OUTPUT_DIR/"df_observations.xlsx", index=False)
print("Parsed sheets:", len(sites_dict))
print("Total observations:", len(df_observations))


Parsed sheets: 10
Total observations: 680


In [6]:
for site, info in sites_dict.items():
    for prof_name, dfp in info['profiles'].items():
        if dfp.empty:
            continue
        safe_site = re.sub(r'[^\w\d\-]', '_', site)[:60]
        safe_prof = re.sub(r'[^\w\d\-]', '_', prof_name)[:80]
        out_path = OUTPUT_DIR / "profiles" / f"{safe_site}__{safe_prof}.xlsx"
        try:
            dfp.to_excel(out_path, index=False)
        except Exception as e:
            print("Save error:", out_path, e)
print("Profiles exported to:", (OUTPUT_DIR / "profiles").resolve())


Profiles exported to: C:\Users\user\Desktop\Diplom\po\results_models\profiles


In [7]:
def plot_profile_linear(dfp, out_png_path, ycol='ИПН_м'):
    # если нет ycol — пробуем РГП_ПН_м
    if ycol not in dfp.columns or dfp[ycol].notna().sum() < 2:
        if 'РГП_ПН_м' in dfp.columns and dfp['РГП_ПН_м'].notna().sum() >= 2:
            ycol = 'РГП_ПН_м'
        else:
            return False
    d = dfp.dropna(subset=['Год', ycol]).sort_values('Год')
    if d.shape[0] < 2:
        return False
    x = d['Год'].values
    y = d[ycol].values
    # линейная регрессия простая polyfit
    coef = np.polyfit(x, y, 1)
    trend = np.poly1d(coef)
    plt.figure(figsize=(7,4))
    plt.plot(d['Год'], d[ycol], 'o-', label='Факт')
    plt.plot(d['Год'], trend(d['Год']), 'r--', label=f'Лин. аппрокс. (slope={coef[0]:.3f} m/yr)')
    plt.title(f"{d.get('Профиль', [''])[0] if 'Профиль' in d.columns else ''}")
    plt.xlabel('Год'); plt.ylabel(f"{ycol} (м)")
    plt.grid(alpha=0.3); plt.legend()
    plt.tight_layout()
    plt.savefig(out_png_path, dpi=150)
    plt.close()
    return True

n_plots = 0
for site, info in sites_dict.items():
    for prof_name, dfp in info['profiles'].items():
        if dfp.empty: continue
        safe_site = re.sub(r'[^\w\d\-]', '_', site)[:60]
        safe_prof = re.sub(r'[^\w\d\-]', '_', prof_name)[:80]
        png_path = OUTPUT_DIR / "plots" / f"{safe_site}__{safe_prof}.png"
        ok = plot_profile_linear(dfp, png_path, ycol='ИПН_м')
        if ok: n_plots += 1
print("Plots saved:", n_plots, "->", (OUTPUT_DIR/"plots").resolve())


Plots saved: 33 -> C:\Users\user\Desktop\Diplom\po\results_models\plots


In [8]:
def compute_slope_retreat(years, values):
    years = np.array(years, dtype=float)
    values = np.array(values, dtype=float)
    mask = ~np.isnan(years) & ~np.isnan(values)
    if mask.sum() < 2:
        return np.nan
    slope = np.polyfit(years[mask], values[mask], 1)[0]
    return -float(slope)  # положительная -> отступание м/год

rows = []
for site, info in sites_dict.items():
    for prof_name, dfp in info['profiles'].items():
        if dfp.empty: continue
        # выбрать ycol предпочтительно ИПН_м
        if 'ИПН_м' in dfp.columns and dfp['ИПН_м'].notna().sum() >= 2:
            ycol = 'ИПН_м'
        elif 'РГП_ПН_м' in dfp.columns and dfp['РГП_ПН_м'].notna().sum() >= 2:
            ycol = 'РГП_ПН_м'
        else:
            ycol = None
        retreat = compute_slope_retreat(dfp['Год'], dfp[ycol]) if ycol else np.nan
        mean_y = float(dfp[ycol].mean()) if ycol and dfp[ycol].notna().any() else np.nan
        std_y = float(dfp[ycol].std()) if ycol and dfp[ycol].notna().any() else np.nan
        duration = int(dfp['Год'].max() - dfp['Год'].min()) if dfp['Год'].notna().any() else np.nan
        rows.append({
            'Участок': site,
            'Профиль': prof_name,
            'n_obs': int(len(dfp)),
            'year_first': int(dfp['Год'].min()),
            'year_last': int(dfp['Год'].max()),
            'duration_years': duration,
            'ycol_used': ycol,
            'mean_y': mean_y,
            'std_y': std_y,
            'retreat_m_per_year': float(retreat) if not np.isnan(retreat) else np.nan
        })

df_profiles = pd.DataFrame(rows)
df_profiles.to_excel(OUTPUT_DIR/"df_profiles_full.xlsx", index=False)
print("Profile summary saved:", OUTPUT_DIR/"df_profiles_full.xlsx")
display(df_profiles.head(12))


Profile summary saved: results_models\df_profiles_full.xlsx


,Участок,Профиль,n_obs,year_first,year_last,duration_years,ycol_used,mean_y,std_y,retreat_m_per_year
0,Бережновка,ПРОФИЛЬ № 60,24,1958,2023,65,РГП_ПН_м,64.126250,60.865202,2.599662
1,Бережновка,ПРОФИЛЬ № 61,24,1958,2023,65,РГП_ПН_м,44.537917,60.304533,2.363369
2,Бережновка,ПРОФИЛЬ № 62,24,1958,2023,65,РГП_ПН_м,81.347273,69.140327,9.087350
3,Бурты,ПРОФИЛЬ № 1,25,1987,2023,36,РГП_ПН_м,43.148400,21.095159,1.873765
4,Бурты,ПРОФИЛЬ № 2,25,1987,2023,36,РГП_ПН_м,35.272000,17.238237,0.790677
5,Бурты,ПРОФИЛЬ № 3,25,1987,2023,36,РГП_ПН_м,12.283333,24.709647,2.078205
6,Молчановка,ПРОФИЛЬ № 57,24,1958,2022,64,РГП_ПН_м,-7.700000,55.416149,-0.939590
7,Молчановка,ПРОФИЛЬ № 58,24,1958,2022,64,РГП_ПН_м,-29.175000,80.780052,-2.716757
8,Молчановка,ПРОФИЛЬ № 59,24,1958,2022,64,РГП_ПН_м,10.551304,54.949932,-0.229817
9,Нижний Балыклей,ПРОФИЛЬ № 52,25,1958,2024,66,РГП_ПН_м,52.955714,45.900058,2.039440


In [9]:
# фильтруем профили с target
ml_df = df_profiles.dropna(subset=['retreat_m_per_year']).copy()
print("Profiles with target:", len(ml_df))
if len(ml_df) < 4:
    print("Внимание: очень мало профилей с рассчитанным target (<4). Модель будет ненадёжной, но мы всё равно выполняем демонстрационное обучение.")

# признаки: n_obs, duration_years, year_first, year_last, mean_y, std_y
feat_cols = ['n_obs','duration_years','year_first','year_last','mean_y','std_y']
X = ml_df[feat_cols].fillna(0).astype(float)
y = ml_df['retreat_m_per_year'].astype(float)

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# pipeline scaler + MLP
mlp = MLPRegressor(hidden_layer_sizes=(32,16), activation='relu', solver='adam', max_iter=1000, random_state=42)
pipe = Pipeline([('scaler', StandardScaler()), ('mlp', mlp)])
pipe.fit(X_train, y_train)

# predict + metrics
y_pred = pipe.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

metrics = {
    'n_profiles': len(ml_df),
    'n_train': len(X_train),
    'n_test': len(X_test),
    'RMSE': float(rmse),
    'MAE': float(mae),
    'R2': float(r2)
}
print("ML metrics:", metrics)

# Сохраним модель и метрики
model_path = OUTPUT_DIR / "models" / "mlp_profiles.joblib"
joblib.dump(pipe, model_path)
(pd.DataFrame([metrics]).to_excel(OUTPUT_DIR/"ml_metrics_profiles.xlsx", index=False))
print("Model saved:", model_path)
print("Metrics saved:", OUTPUT_DIR/"ml_metrics_profiles.xlsx")


Profiles with target: 33
ML metrics: {'n_profiles': 33, 'n_train': 26, 'n_test': 7, 'RMSE': 0.8979594782094228, 'MAE': 0.6992651947281232, 'R2': 0.39433165003289294}
Model saved: results_models\models\mlp_profiles.joblib
Metrics saved: results_models\ml_metrics_profiles.xlsx


In [10]:
print("Готово.")
print("— profiles exported in:", (OUTPUT_DIR/"profiles").resolve())
print("— profile plots in:", (OUTPUT_DIR/"plots").resolve())
print("— observations table:", (OUTPUT_DIR/"df_observations.xlsx").resolve())
print("— profiles summary:", (OUTPUT_DIR/"df_profiles_full.xlsx").resolve())
print("— trained model (if any):", (OUTPUT_DIR/"models"/"mlp_profiles.joblib").resolve())
print("— ml metrics:", (OUTPUT_DIR/"ml_metrics_profiles.xlsx").resolve())


Готово.
— profiles exported in: C:\Users\user\Desktop\Diplom\po\results_models\profiles
— profile plots in: C:\Users\user\Desktop\Diplom\po\results_models\plots
— observations table: C:\Users\user\Desktop\Diplom\po\results_models\df_observations.xlsx
— profiles summary: C:\Users\user\Desktop\Diplom\po\results_models\df_profiles_full.xlsx
— trained model (if any): C:\Users\user\Desktop\Diplom\po\results_models\models\mlp_profiles.joblib
— ml metrics: C:\Users\user\Desktop\Diplom\po\results_models\ml_metrics_profiles.xlsx
